# BurnOmicsDB: NCBI Human Gene Search Index v2

This Notebook reads `Homo_sapiens.gene_info.gz`, retains `Homo sapiens` records (`tax_id = 9606`), and produces two clearly separated output groups.

## Website/search outputs

1. `human_gene_alias_search_index.json`  
   Website source containing:
   - a gene master indexed by NCBI GeneID;
   - a search index mapping normalized identifiers, symbols, aliases and gene names to one or more NCBI GeneIDs.

2. `human_gene_aliases.xlsx`  
   A single-sheet human-readable gene reference table containing only gene identifiers and search/display fields. It contains no statistics.

## Audit output

3. `human_gene_alias_statistics.txt`  
   Processing metadata, summary statistics, field completeness, gene-type counts, chromosome counts and ambiguous-term summaries.

## Supported search inputs

- NCBI GeneID, such as `7157`
- official gene symbol, such as `TP53`
- NCBI synonym or historical alias, such as `P53`
- NCBI gene description/full name, such as `tumor protein p53`
- NCBI nomenclature symbol and full name
- Ensembl Gene ID, such as `ENSG00000141510`
- HGNC ID, such as `HGNC:11998`

Search terms are normalized by Unicode NFKC normalization, trimming, collapsing repeated whitespace, removing spaces around colons, and converting to uppercase. Ambiguous terms retain all candidate NCBI GeneIDs.

`Other_designations` is retained in the Excel gene reference but is not indexed for public search because NCBI may use broad or automatically generated descriptions that map to many unrelated genes.

## Recommended search identifiers

Users should preferentially search with an **NCBI GeneID** or an **official gene symbol**. NCBI GeneIDs are the most stable and unambiguous identifiers. Official symbols are standardized and human-readable. Aliases, historical symbols and full gene names are also indexed for accessibility, but some terms can refer to multiple genes. BurnOmicsDB must display all candidate genes for an ambiguous query rather than selecting one silently.

In [1]:
# ---- 00. Package installation / 依赖安装 ----
# Install only missing packages. Console messages and output files use English.

import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "xlsxwriter": "XlsxWriter",
}

for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package_name]
        )

print("Package check completed.")

Package check completed.


In [2]:
# ---- 01. Project settings / 项目设置 ----

from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import re
import unicodedata

import pandas as pd
from IPython.display import display

SOURCE_URL = (
    "https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/"
    "Mammalia/Homo_sapiens.gene_info.gz"
)
SOURCE_LAST_MODIFIED = "2026-07-24 19:05"
TARGET_TAX_ID = "9606"

WORK_DIR = Path.cwd()

# These names replace the previous JSON and Excel, while adding a new TXT report.
OUTPUT_JSON = WORK_DIR / "human_gene_alias_search_index.json"
OUTPUT_EXCEL = WORK_DIR / "human_gene_aliases.xlsx"
OUTPUT_STATS = WORK_DIR / "human_gene_alias_statistics.txt"

GENERATED_AT_UTC = datetime.now(timezone.utc).isoformat()

print(f"Working directory: {WORK_DIR}")
print(f"Target species: Homo sapiens (tax_id={TARGET_TAX_ID})")
print("Output files:")
print(f" - {OUTPUT_JSON.name}")
print(f" - {OUTPUT_EXCEL.name}")
print(f" - {OUTPUT_STATS.name}")

Working directory: /Users/peter/Downloads/Project-2026-BurnOmicsDB/Gene_Aliases
Target species: Homo sapiens (tax_id=9606)
Output files:
 - human_gene_alias_search_index.json
 - human_gene_aliases.xlsx
 - human_gene_alias_statistics.txt


In [3]:
# ---- 02. Locate the NCBI input file / 自动定位输入文件 ----

preferred_file = WORK_DIR / "Homo_sapiens.gene_info.gz"

if preferred_file.exists():
    input_file = preferred_file
else:
    candidates = sorted(
        WORK_DIR.glob("Homo_sapiens.gene_info*.gz"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            "No Homo_sapiens.gene_info*.gz file was found in the current "
            "directory. Place the Notebook and the NCBI gzip file in the "
            "same Gene_Aliases directory."
        )
    input_file = candidates[0]
    if len(candidates) > 1:
        print("Multiple candidate files were found. The newest file will be used:")
        for candidate in candidates:
            marker = " [selected]" if candidate == input_file else ""
            print(f" - {candidate.name}{marker}")

print(f"Input file: {input_file.name}")
print(f"Compressed size: {input_file.stat().st_size / 1024 / 1024:.2f} MB")

Input file: Homo_sapiens.gene_info.gz
Compressed size: 4.93 MB


In [4]:
# ---- 03. Read and validate NCBI gene_info / 读取并验证原始数据 ----

raw_df = pd.read_csv(
    input_file,
    sep="\t",
    compression="gzip",
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)

raw_df = raw_df.rename(columns={"#tax_id": "tax_id"})

required_columns = {
    "tax_id",
    "GeneID",
    "Symbol",
    "Synonyms",
    "dbXrefs",
    "chromosome",
    "map_location",
    "description",
    "type_of_gene",
    "Symbol_from_nomenclature_authority",
    "Full_name_from_nomenclature_authority",
    "Nomenclature_status",
    "Other_designations",
    "Modification_date",
}

missing_columns = sorted(required_columns - set(raw_df.columns))
if missing_columns:
    raise ValueError(
        "The NCBI input file is missing required columns: "
        + ", ".join(missing_columns)
    )

human_df = raw_df.loc[raw_df["tax_id"] == TARGET_TAX_ID].copy()

if human_df.empty:
    raise RuntimeError(
        f"No records remained after filtering for tax_id={TARGET_TAX_ID}."
    )

print(f"Raw records: {len(raw_df):,}")
print(f"Raw columns: {raw_df.shape[1]:,}")
print(f"Homo sapiens records: {len(human_df):,}")
print(f"Excluded non-human records: {len(raw_df) - len(human_df):,}")
display(human_df.head(3))

Raw records: 193,884
Raw columns: 16
Homo sapiens records: 193,811
Excluded non-human records: 73


,tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type
0,9606,1,A1BG,-,A1B|ABG|GAB|HYST2477,MIM:138670|HGNC:HGNC:5|Ensembl:ENSG00000121410...,19,19q13.43,alpha-1-B glycoprotein,protein-coding,A1BG,alpha-1-B glycoprotein,O,alpha-1B-glycoprotein|HEL-S-163pA|epididymis s...,20260723,-
1,9606,2,A2M,-,A2MD|CPAMD5|FWP007|S863-7,MIM:103950|HGNC:HGNC:7|Ensembl:ENSG00000175899...,12,12p13.31,alpha-2-macroglobulin,protein-coding,A2M,alpha-2-macroglobulin,O,alpha-2-macroglobulin|C3 and PZP-like alpha-2-...,20260706,-
2,9606,9,NAT1,-,AAC1|MNAT|NAT-1|NATI,MIM:108345|HGNC:HGNC:7645|Ensembl:ENSG00000171...,8,8p22,N-acetyltransferase 1,protein-coding,NAT1,N-acetyltransferase 1,O,arylamine N-acetyltransferase 1|N-acetyltransf...,20260706,-


In [5]:
# ---- 04. Cleaning and helper functions / 字段清理与辅助函数 ----

MISSING_MARKERS = {"", "-", "nan", "None", "NA", "N/A"}
MISSING_MARKERS_NORMALIZED = {
    marker.upper() for marker in MISSING_MARKERS
}


def clean_ncbi_value(value):
    """Convert NCBI missing markers to an empty string."""
    text = str(value).strip()
    return "" if text in MISSING_MARKERS else text


def normalize_search_term(value):
    """Normalize a user-searchable term without deleting meaningful punctuation."""
    text = clean_ncbi_value(value)
    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text.strip())
    text = re.sub(r"\s*:\s*", ":", text)
    return text.upper()


def extract_dbxref(dbxrefs, database_name):
    """Extract the first requested database identifier from NCBI dbXrefs."""
    text = clean_ncbi_value(dbxrefs)
    if not text:
        return ""

    prefix = f"{database_name}:"
    for item in text.split("|"):
        item = item.strip()
        if item.startswith(prefix):
            return item.split(":", 1)[1]
    return ""


def sha256_file(path, chunk_size=1024 * 1024):
    """Calculate a SHA-256 checksum without loading the entire file."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def normalized_series(series):
    """Vectorized equivalent of normalize_search_term for a pandas Series."""
    output = (
        series.fillna("")
        .astype(str)
        .str.normalize("NFKC")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"\s*:\s*", ":", regex=True)
        .str.upper()
    )
    return output.where(
        ~output.isin(MISSING_MARKERS_NORMALIZED), ""
    )

In [6]:
# ---- 05. Build the website gene master / 构建网站基因主表 ----

human_df["HGNC_ID"] = human_df["dbXrefs"].map(
    lambda value: extract_dbxref(value, "HGNC")
)
human_df["Ensembl_Gene_ID"] = human_df["dbXrefs"].map(
    lambda value: extract_dbxref(value, "Ensembl")
)

gene_master_df = pd.DataFrame({
    "NCBI_GeneID": human_df["GeneID"].map(clean_ncbi_value),
    "Official_Symbol": human_df["Symbol"].map(clean_ncbi_value),
    "Gene_Name": human_df["description"].map(clean_ncbi_value),
    "Gene_Type": human_df["type_of_gene"].map(clean_ncbi_value),
    "Synonyms": human_df["Synonyms"].map(clean_ncbi_value),
    "Nomenclature_Symbol": human_df[
        "Symbol_from_nomenclature_authority"
    ].map(clean_ncbi_value),
    "Nomenclature_Full_Name": human_df[
        "Full_name_from_nomenclature_authority"
    ].map(clean_ncbi_value),
    "Other_Designations": human_df[
        "Other_designations"
    ].map(clean_ncbi_value),
    "Ensembl_Gene_ID": human_df["Ensembl_Gene_ID"].map(clean_ncbi_value),
    "HGNC_ID": human_df["HGNC_ID"].map(clean_ncbi_value),
})

if gene_master_df["NCBI_GeneID"].eq("").any():
    raise ValueError("At least one human record has a missing NCBI GeneID.")

duplicated_gene_ids = gene_master_df.loc[
    gene_master_df["NCBI_GeneID"].duplicated(keep=False),
    "NCBI_GeneID",
].unique()

if len(duplicated_gene_ids) > 0:
    raise ValueError(
        "Duplicated NCBI GeneIDs were detected: "
        + ", ".join(duplicated_gene_ids[:20])
    )

gene_master_df["_GeneID_Numeric"] = pd.to_numeric(
    gene_master_df["NCBI_GeneID"], errors="raise"
)
gene_master_df = (
    gene_master_df
    .sort_values("_GeneID_Numeric", kind="stable")
    .drop(columns="_GeneID_Numeric")
    .reset_index(drop=True)
)

print(f"Gene master records: {len(gene_master_df):,}")
print(
    "Unique official symbols: "
    f"{gene_master_df['Official_Symbol'].nunique():,}"
)
display(gene_master_df.head(5))

Gene master records: 193,811
Unique official symbols: 193,708


,NCBI_GeneID,Official_Symbol,Gene_Name,Gene_Type,Synonyms,Nomenclature_Symbol,Nomenclature_Full_Name,Other_Designations,Ensembl_Gene_ID,HGNC_ID
0,1,A1BG,alpha-1-B glycoprotein,protein-coding,A1B|ABG|GAB|HYST2477,A1BG,alpha-1-B glycoprotein,alpha-1B-glycoprotein|HEL-S-163pA|epididymis s...,ENSG00000121410,HGNC:5
1,2,A2M,alpha-2-macroglobulin,protein-coding,A2MD|CPAMD5|FWP007|S863-7,A2M,alpha-2-macroglobulin,alpha-2-macroglobulin|C3 and PZP-like alpha-2-...,ENSG00000175899,HGNC:7
2,9,NAT1,N-acetyltransferase 1,protein-coding,AAC1|MNAT|NAT-1|NATI,NAT1,N-acetyltransferase 1,arylamine N-acetyltransferase 1|N-acetyltransf...,ENSG00000171428,HGNC:7645
3,10,NAT2,N-acetyltransferase 2,protein-coding,AAC2|NAT-2|PNAT,NAT2,N-acetyltransferase 2,arylamine N-acetyltransferase 2|N-acetyltransf...,ENSG00000156006,HGNC:7646
4,11,NATP,N-acetyltransferase pseudogene,pseudo,AACP|NATP1,NATP,N-acetyltransferase pseudogene,arylamide acetylase pseudogene,,HGNC:15


In [7]:
# ---- 06. Build the complete search-term mapping / 构建完整检索映射 ----
# Each search term maps to an NCBI GeneID. Ambiguous matches are retained.

TERM_TYPE_PRIORITY = {
    "ncbi_gene_id": 0,
    "official_symbol": 1,
    "ensembl_gene_id": 2,
    "hgnc_id": 3,
    "nomenclature_symbol": 4,
    "gene_name": 5,
    "nomenclature_full_name": 6,
    "synonym": 7,
}


def make_single_term_frame(source_column, search_type):
    if source_column == "NCBI_GeneID":
        frame = gene_master_df[["NCBI_GeneID"]].copy()
        frame["Search_Term"] = frame["NCBI_GeneID"]
    else:
        frame = gene_master_df[["NCBI_GeneID", source_column]].copy()
        frame = frame.rename(columns={source_column: "Search_Term"})

    frame["Search_Term"] = frame["Search_Term"].map(clean_ncbi_value)
    frame["Search_Type"] = search_type
    return frame


def make_pipe_term_frame(source_column, search_type):
    frame = gene_master_df[["NCBI_GeneID", source_column]].copy()
    frame["Search_Term"] = (
        frame[source_column]
        .fillna("")
        .astype(str)
        .str.split("|", regex=False)
    )
    frame = frame.explode("Search_Term", ignore_index=True)
    frame["Search_Term"] = (
        frame["Search_Term"]
        .fillna("")
        .astype(str)
        .str.strip()
    )
    frame = frame.drop(columns=source_column)
    frame["Search_Type"] = search_type
    return frame


term_frames = [
    make_single_term_frame("NCBI_GeneID", "ncbi_gene_id"),
    make_single_term_frame("Official_Symbol", "official_symbol"),
    make_single_term_frame("Ensembl_Gene_ID", "ensembl_gene_id"),
    make_single_term_frame("HGNC_ID", "hgnc_id"),
    make_single_term_frame("Nomenclature_Symbol", "nomenclature_symbol"),
    make_single_term_frame("Gene_Name", "gene_name"),
    make_single_term_frame(
        "Nomenclature_Full_Name", "nomenclature_full_name"
    ),
    make_pipe_term_frame("Synonyms", "synonym"),
]

search_mapping_df = pd.concat(term_frames, ignore_index=True)
search_mapping_df["Search_Term"] = (
    search_mapping_df["Search_Term"]
    .fillna("")
    .astype(str)
    .str.strip()
)
search_mapping_df["Search_Term_Normalized"] = normalized_series(
    search_mapping_df["Search_Term"]
)

search_mapping_df = search_mapping_df.loc[
    search_mapping_df["Search_Term_Normalized"].ne("")
].copy()

search_mapping_df["_Search_Type_Priority"] = (
    search_mapping_df["Search_Type"].map(TERM_TYPE_PRIORITY)
)
search_mapping_df["_GeneID_Numeric"] = pd.to_numeric(
    search_mapping_df["NCBI_GeneID"], errors="raise"
)

# The same term may occur in multiple NCBI fields for the same gene.
# Retain one term-gene pair, prioritizing the most specific identifier type.
search_mapping_df = (
    search_mapping_df
    .sort_values(
        [
            "Search_Term_Normalized",
            "_GeneID_Numeric",
            "_Search_Type_Priority",
        ],
        kind="stable",
    )
    .drop_duplicates(
        subset=["Search_Term_Normalized", "NCBI_GeneID"],
        keep="first",
    )
    .drop(columns=["_Search_Type_Priority", "_GeneID_Numeric"])
    .reset_index(drop=True)
)

print(f"Unique term-gene pairs: {len(search_mapping_df):,}")
print(
    "Unique normalized search terms: "
    f"{search_mapping_df['Search_Term_Normalized'].nunique():,}"
)
display(search_mapping_df.head(10))

Unique term-gene pairs: 739,036
Unique normalized search terms: 730,234


,NCBI_GeneID,Search_Term,Search_Type,Search_Term_Normalized
0,3845,'C-K-RAS,synonym,'C-K-RAS
1,10316,(FM-3),synonym,(FM-3)
2,28337,(IV)-44,synonym,(IV)-44
3,10159,(P)RR,synonym,(P)RR
4,374659,(ppGpp)ase,synonym,(PPGPP)ASE
5,110599572,-8 kb enhancer of CYP2C8,gene_name,-8 KB ENHANCER OF CYP2C8
6,79585,0610011B16Rik,synonym,0610011B16RIK
7,10248,0610037N12Rik,synonym,0610037N12RIK
8,10975,0710008D09Rik,synonym,0710008D09RIK
9,100130557,0808y08y,synonym,0808Y08Y


## JSON structure

The JSON avoids repeating complete gene annotations under every alias.

```json
{
  "metadata": {
    "gene_record_schema": [
      "official_symbol",
      "gene_name",
      "gene_type",
      "synonyms",
      "nomenclature_symbol",
      "nomenclature_full_name",
      "ensembl_gene_id",
      "hgnc_id"
    ]
  },
  "genes": {
    "7157": [
      "TP53",
      "tumor protein p53",
      "protein-coding",
      "BCC7|LFS1|P53|TRP53",
      "TP53",
      "tumor protein p53",
      "ENSG00000141510",
      "HGNC:11998"
    ]
  },
  "search": {
    "7157": ["7157"],
    "P53": ["7157"],
    "TUMOR PROTEIN P53": ["7157"]
  }
}
```

Every search value is a list. This preserves ambiguous terms safely.

In [8]:
# ---- 07. Build and save the website JSON / 构建并保存网站JSON ----

GENE_RECORD_COLUMNS = [
    "Official_Symbol",
    "Gene_Name",
    "Gene_Type",
    "Synonyms",
    "Nomenclature_Symbol",
    "Nomenclature_Full_Name",
    "Ensembl_Gene_ID",
    "HGNC_ID",
]

gene_index = {
    gene_id: list(values)
    for gene_id, *values in gene_master_df[
        ["NCBI_GeneID", *GENE_RECORD_COLUMNS]
    ].itertuples(index=False, name=None)
}

search_index = defaultdict(list)
for search_term, gene_id in search_mapping_df[
    ["Search_Term_Normalized", "NCBI_GeneID"]
].itertuples(index=False, name=None):
    search_index[search_term].append(gene_id)

ambiguous_search_terms = {
    term: gene_ids
    for term, gene_ids in search_index.items()
    if len(gene_ids) > 1
}

json_payload = {
    "metadata": {
        "project": "BurnOmicsDB",
        "database": "NCBI Gene",
        "species": "Homo sapiens",
        "tax_id": TARGET_TAX_ID,
        "source_url": SOURCE_URL,
        "source_last_modified": SOURCE_LAST_MODIFIED,
        "input_file": input_file.name,
        "generated_at_utc": GENERATED_AT_UTC,
        "search_normalization": (
            "Unicode NFKC; trim; collapse whitespace; remove spaces around "
            "colons; convert to uppercase"
        ),
        "supported_search_types": list(TERM_TYPE_PRIORITY.keys()),
        "gene_record_schema": [
            "official_symbol",
            "gene_name",
            "gene_type",
            "synonyms",
            "nomenclature_symbol",
            "nomenclature_full_name",
            "ensembl_gene_id",
            "hgnc_id",
        ],
        "search_record_schema": ["ncbi_gene_id"],
        "ambiguity_policy": (
            "A search term always maps to a list. All candidate NCBI GeneIDs "
            "are retained when a term is ambiguous."
        ),
    },
    "genes": gene_index,
    "search": dict(search_index),
}

with OUTPUT_JSON.open("w", encoding="utf-8") as file_handle:
    json.dump(
        json_payload,
        file_handle,
        ensure_ascii=False,
        separators=(",", ":"),
    )

print(f"JSON created: {OUTPUT_JSON.name}")
print(f"JSON size: {OUTPUT_JSON.stat().st_size / 1024 / 1024:.2f} MB")
print(f"Ambiguous search terms: {len(ambiguous_search_terms):,}")

JSON created: human_gene_alias_search_index.json
JSON size: 47.51 MB
Ambiguous search terms: 5,222


In [9]:
# ---- 08. Save the website/reference Excel / 输出精简Excel ----
# The workbook contains only Gene_Master. Statistics go to the TXT report.

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="xlsxwriter",
    engine_kwargs={"options": {"strings_to_urls": False}},
) as writer:

    gene_master_df.to_excel(
        writer,
        sheet_name="Gene_Master",
        index=False,
    )

    workbook = writer.book
    worksheet = writer.sheets["Gene_Master"]

    header_format = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
    })
    text_format = workbook.add_format({
        "text_wrap": True,
        "valign": "top",
    })

    worksheet.freeze_panes(1, 2)
    worksheet.autofilter(
        0,
        0,
        len(gene_master_df),
        len(gene_master_df.columns) - 1,
    )
    worksheet.set_row(0, 24)

    for column_index, column_name in enumerate(gene_master_df.columns):
        worksheet.write(0, column_index, column_name, header_format)

    column_widths = {
        "NCBI_GeneID": 14,
        "Official_Symbol": 22,
        "Gene_Name": 48,
        "Gene_Type": 20,
        "Synonyms": 55,
        "Nomenclature_Symbol": 24,
        "Nomenclature_Full_Name": 55,
        "Other_Designations": 70,
        "Ensembl_Gene_ID": 22,
        "HGNC_ID": 16,
    }

    for column_index, column_name in enumerate(gene_master_df.columns):
        width = column_widths.get(column_name, 20)
        cell_format = text_format if column_name in {
            "Gene_Name",
            "Synonyms",
            "Nomenclature_Full_Name",
            "Other_Designations",
        } else None
        worksheet.set_column(
            column_index,
            column_index,
            width,
            cell_format,
        )

print(f"Excel created: {OUTPUT_EXCEL.name}")
print(f"Excel size: {OUTPUT_EXCEL.stat().st_size / 1024 / 1024:.2f} MB")

Excel created: human_gene_aliases.xlsx
Excel size: 9.35 MB


In [10]:
# ---- 09. Create the statistics TXT report / 输出独立统计报告 ----

gene_type_counts = (
    gene_master_df["Gene_Type"]
    .replace("", "Missing")
    .value_counts(dropna=False)
)

chromosome_counts = (
    human_df["chromosome"]
    .map(clean_ncbi_value)
    .replace("", "Missing")
    .value_counts(dropna=False)
)

term_type_counts = (
    search_mapping_df["Search_Type"]
    .value_counts()
    .reindex(TERM_TYPE_PRIORITY.keys(), fill_value=0)
)

term_gene_counts = (
    search_mapping_df
    .groupby("Search_Term_Normalized")["NCBI_GeneID"]
    .nunique()
    .sort_values(ascending=False)
)

ambiguous_term_count = int((term_gene_counts > 1).sum())
maximum_genes_per_term = int(term_gene_counts.max())
top_ambiguous_terms = term_gene_counts.loc[
    term_gene_counts > 1
].head(100)

missing_field_counts = {
    column_name: int(gene_master_df[column_name].eq("").sum())
    for column_name in gene_master_df.columns
}

input_sha256 = sha256_file(input_file)
json_sha256 = sha256_file(OUTPUT_JSON)
excel_sha256 = sha256_file(OUTPUT_EXCEL)

report_lines = [
    "BurnOmicsDB Human Gene Search Index - Processing Statistics",
    "=" * 66,
    "",
    "SOURCE AND GENERATION",
    "Project: BurnOmicsDB",
    "Database: NCBI Gene",
    "Species: Homo sapiens",
    f"Tax ID: {TARGET_TAX_ID}",
    f"Source URL: {SOURCE_URL}",
    f"Source last modified: {SOURCE_LAST_MODIFIED}",
    f"Input file: {input_file.name}",
    f"Input file size (MB): {input_file.stat().st_size / 1024 / 1024:.2f}",
    f"Input SHA-256: {input_sha256}",
    f"Generated at UTC: {GENERATED_AT_UTC}",
    "",
    "CORE RECORD COUNTS",
    f"Raw record count: {len(raw_df):,}",
    f"Homo sapiens record count: {len(human_df):,}",
    f"Excluded non-human record count: {len(raw_df) - len(human_df):,}",
    f"Unique NCBI GeneID count: {gene_master_df['NCBI_GeneID'].nunique():,}",
    f"Unique official symbol count: {gene_master_df['Official_Symbol'].nunique():,}",
    f"Protein-coding gene count: {(gene_master_df['Gene_Type'] == 'protein-coding').sum():,}",
    "",
    "SEARCH INDEX COUNTS",
    f"Unique term-gene pair count: {len(search_mapping_df):,}",
    f"Unique normalized search term count: {search_mapping_df['Search_Term_Normalized'].nunique():,}",
    f"Ambiguous search term count: {ambiguous_term_count:,}",
    f"Maximum genes mapped by one search term: {maximum_genes_per_term:,}",
    "",
    "SEARCH TERM COUNTS BY RETAINED TYPE",
]

for search_type, count in term_type_counts.items():
    report_lines.append(f"{search_type}: {int(count):,}")

report_lines.extend(["", "FIELD COMPLETENESS"])
for column_name, missing_count in missing_field_counts.items():
    report_lines.append(
        f"{column_name} missing: {missing_count:,} "
        f"({missing_count / len(gene_master_df) * 100:.3f}%)"
    )

report_lines.extend(["", "GENE TYPE COUNTS"])
for gene_type, count in gene_type_counts.items():
    report_lines.append(f"{gene_type}: {int(count):,}")

report_lines.extend(["", "CHROMOSOME COUNTS"])
for chromosome, count in chromosome_counts.items():
    report_lines.append(f"{chromosome}: {int(count):,}")

report_lines.extend([
    "",
    "TOP 100 AMBIGUOUS NORMALIZED SEARCH TERMS",
    "Search_Term_Normalized\tGene_Count\tNCBI_GeneIDs",
])
for term, gene_count in top_ambiguous_terms.items():
    report_lines.append(
        f"{term}\t{int(gene_count)}\t{'|'.join(search_index[term])}"
    )

report_lines.extend([
    "",
    "OUTPUT FILES",
    f"{OUTPUT_JSON.name} size (MB): {OUTPUT_JSON.stat().st_size / 1024 / 1024:.2f}",
    f"{OUTPUT_JSON.name} SHA-256: {json_sha256}",
    f"{OUTPUT_EXCEL.name} size (MB): {OUTPUT_EXCEL.stat().st_size / 1024 / 1024:.2f}",
    f"{OUTPUT_EXCEL.name} SHA-256: {excel_sha256}",
    "",
    "NOTES",
    (
        "The Excel workbook contains only the Gene_Master sheet and is intended "
        "for human inspection and reference."
    ),
    (
        "The JSON search index includes NCBI GeneIDs, official symbols, "
        "nomenclature symbols, gene names, aliases, Ensembl Gene IDs and "
        "HGNC IDs."
    ),
    (
        "NCBI Other_designations is retained in the Excel reference but excluded "
        "from public search because broad automated phrases can map to many genes."
    ),
    (
        "Ambiguous search terms retain every candidate NCBI GeneID. The website "
        "must ask the user to choose rather than selecting one automatically."
    ),
    (
        "The complete JSON is a website source artifact. It may later be split "
        "into smaller chunks for efficient browser loading."
    ),
])

OUTPUT_STATS.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

print(f"Statistics report created: {OUTPUT_STATS.name}")
print(f"Statistics report size: {OUTPUT_STATS.stat().st_size / 1024:.2f} KB")

Statistics report created: human_gene_alias_statistics.txt
Statistics report size: 27.40 KB


In [11]:
# ---- 10. Validate representative searches / 验证代表性检索 ----

def search_gene(query):
    normalized_query = normalize_search_term(query)
    matched_gene_ids = json_payload["search"].get(normalized_query, [])

    rows = []
    schema = json_payload["metadata"]["gene_record_schema"]
    for gene_id in matched_gene_ids:
        record = dict(zip(schema, json_payload["genes"][gene_id]))
        rows.append({
            "Query": query,
            "Normalized_Query": normalized_query,
            "NCBI_GeneID": gene_id,
            **record,
        })
    return pd.DataFrame(rows)


test_queries = [
    "TP53",
    "P53",
    "7157",
    "tumor protein p53",
    "ENSG00000141510",
    "HGNC:11998",
    "IL8",
]

for query in test_queries:
    result = search_gene(query)
    print(f"Query: {query} | Matches: {len(result)}")
    display(result.head(10))

with OUTPUT_JSON.open("r", encoding="utf-8") as file_handle:
    validation_json = json.load(file_handle)

if "genes" not in validation_json or "search" not in validation_json:
    raise RuntimeError("The output JSON is missing genes or search sections.")
if validation_json["search"].get("7157") != ["7157"]:
    raise RuntimeError("Numeric NCBI GeneID search validation failed for 7157.")
if validation_json["search"].get("TUMOR PROTEIN P53") != ["7157"]:
    raise RuntimeError("Full gene-name search validation failed for TP53.")
if validation_json["search"].get("P53") != ["7157"]:
    raise RuntimeError("Alias search validation failed for P53.")
if validation_json["search"].get("HGNC:11998") != ["7157"]:
    raise RuntimeError("HGNC ID search validation failed for TP53.")

print("All representative search and output validations passed.")

Query: TP53 | Matches: 1


,Query,Normalized_Query,NCBI_GeneID,official_symbol,gene_name,gene_type,synonyms,nomenclature_symbol,nomenclature_full_name,ensembl_gene_id,hgnc_id
0,TP53,TP53,7157,TP53,tumor protein p53,protein-coding,BCC7|BMFS5|LFS1|P53|TRP53,TP53,tumor protein p53,ENSG00000141510,HGNC:11998


Query: P53 | Matches: 1


,Query,Normalized_Query,NCBI_GeneID,official_symbol,gene_name,gene_type,synonyms,nomenclature_symbol,nomenclature_full_name,ensembl_gene_id,hgnc_id
0,P53,P53,7157,TP53,tumor protein p53,protein-coding,BCC7|BMFS5|LFS1|P53|TRP53,TP53,tumor protein p53,ENSG00000141510,HGNC:11998


Query: 7157 | Matches: 1


...

Query: HGNC:11998 | Matches: 1


,Query,Normalized_Query,NCBI_GeneID,official_symbol,gene_name,gene_type,synonyms,nomenclature_symbol,nomenclature_full_name,ensembl_gene_id,hgnc_id
0,HGNC:11998,HGNC:11998,7157,TP53,tumor protein p53,protein-coding,BCC7|BMFS5|LFS1|P53|TRP53,TP53,tumor protein p53,ENSG00000141510,HGNC:11998


Query: IL8 | Matches: 1


,Query,Normalized_Query,NCBI_GeneID,official_symbol,gene_name,gene_type,synonyms,nomenclature_symbol,nomenclature_full_name,ensembl_gene_id,hgnc_id
0,IL8,IL8,3576,CXCL8,C-X-C motif chemokine ligand 8,protein-coding,GCP-1|GCP1|IL8|LECT|LUCT|LYNAP|MDNCF|MONAP|NAF...,CXCL8,C-X-C motif chemokine ligand 8,ENSG00000169429,HGNC:6025


All representative search and output validations passed.


In [12]:
# ---- 11. Final output summary / 最终输出汇总 ----

print("Processing completed.")
print("Website/search outputs:")
print(f" - {OUTPUT_JSON.name}")
print(f" - {OUTPUT_EXCEL.name}")
print("Audit output:")
print(f" - {OUTPUT_STATS.name}")
print("")
print(
    "Recommended website behavior: resolve the user query through the JSON "
    "search section, show all candidates when multiple GeneIDs are returned, "
    "and use the genes section to display the standardized annotation."
)

Processing completed.
Website/search outputs:
 - human_gene_alias_search_index.json
 - human_gene_aliases.xlsx
Audit output:
 - human_gene_alias_statistics.txt

Recommended website behavior: resolve the user query through the JSON search section, show all candidates when multiple GeneIDs are returned, and use the genes section to display the standardized annotation.
